# Gold province reporting-area population fact

## Purpose

Publish authoritative Table C population at the D-55 reporting-area grain without replacing the regional population fact.

## Grain

One reporting area per reference year, authoritative source, and deterministic Silver run.


## 1. Gate validated Silver population and Gold dimensions


In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

DECLARE OR REPLACE VARIABLE fact_rule_version STRING
    DEFAULT 'gold-fact-province-reporting-area-population-v1';

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT(DISTINCT run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT NAMED_STRUCT(
        'area', reporting_area_key, 'year', reference_year, 'source', source_name
    )),
    'STOP: Silver reporting-area population is empty, mixed-run, or duplicated'
)
FROM `02-silver`.silver_province_reporting_area_population;

SELECT ASSERT_TRUE(
    COUNT_IF(reporting_area_key = 0) = 1
    AND COUNT(*) = COUNT(DISTINCT reporting_area_key),
    'STOP: run Gold reporting-area dimensions before this fact'
)
FROM `03-gold`.dim_province_reporting_area;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: Silver reporting-area population validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_province_reporting_area_population'
  AND pipeline_run_id = (
      SELECT MAX(run_id)
      FROM `02-silver`.silver_province_reporting_area_population
  );


## 2. Create, validate, and merge the fact


In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.fact_province_reporting_area_population (
    reporting_area_population_fact_key BIGINT NOT NULL COMMENT 'Deterministic key for reporting area, year, and source.',
    reporting_area_key BIGINT NOT NULL COMMENT 'Joins to dim_province_reporting_area.',
    reference_year INT NOT NULL COMMENT 'Authoritative Table C reference year.',
    population_count BIGINT COMMENT 'Accepted-primary barangay population. NULL when unavailable.',
    source_name STRING NOT NULL COMMENT 'Authoritative population source.',
    is_primary_source BOOLEAN NOT NULL COMMENT 'True for authoritative Table C population.',
    population_match_status STRING NOT NULL COMMENT 'Matched, missing, or unresolved accepted population status.',
    source_row_count BIGINT NOT NULL COMMENT 'Accepted barangay rows contributing to the value.',
    source_snapshot_id STRING NOT NULL COMMENT 'Table C source snapshot.',
    run_id STRING NOT NULL COMMENT 'Deterministic Gold fact run ID.',
    source_load_ts TIMESTAMP COMMENT 'Latest source load time.',
    population_reconciliation_run_id STRING NOT NULL COMMENT 'Silver Table C reconciliation run.',
    reporting_area_population_run_id STRING NOT NULL COMMENT 'Silver reporting-area population run.'
)
USING DELTA
COMMENT 'Authoritative population fact at the D-55 province reporting-area grain.';

CREATE OR REPLACE TEMPORARY VIEW fact_province_reporting_area_population_source AS
SELECT
    XXHASH64(CONCAT_WS(
        '|', 'PROVINCE_REPORTING_AREA_POPULATION',
        CAST(area.reporting_area_key AS STRING),
        CAST(population.reference_year AS STRING), population.source_name
    )) AS reporting_area_population_fact_key,
    area.reporting_area_key, population.reference_year, population.population_count,
    population.source_name, population.is_primary_source,
    population.population_match_status, population.source_row_count,
    population.source_snapshot_id,
    SHA2(CONCAT_WS(
        '||', population.run_id, MAX(area.run_id) OVER (), fact_rule_version
    ), 256) AS run_id,
    population.source_load_ts, population.population_reconciliation_run_id,
    population.run_id AS reporting_area_population_run_id
FROM `02-silver`.silver_province_reporting_area_population AS population
INNER JOIN `03-gold`.dim_province_reporting_area AS area
    ON population.reporting_area_key = area.reporting_area_key
   AND population.psgc_version <=> area.psgc_version;

SELECT ASSERT_TRUE(
    COUNT(*) = (
        SELECT COUNT(*)
        FROM `02-silver`.silver_province_reporting_area_population
    )
    AND COUNT(*) = COUNT(DISTINCT reporting_area_population_fact_key)
    AND COUNT(*) = COUNT(DISTINCT NAMED_STRUCT(
        'area', reporting_area_key, 'year', reference_year, 'source', source_name
    ))
    AND SUM(population_count) <=> (
        SELECT SUM(population_count)
        FROM `02-silver`.silver_province_reporting_area_population
    ),
    'STOP: Gold reporting-area population grain, keys, or total do not reconcile'
)
FROM fact_province_reporting_area_population_source;

MERGE INTO `03-gold`.fact_province_reporting_area_population AS target
USING fact_province_reporting_area_population_source AS source
ON target.reporting_area_population_fact_key = source.reporting_area_population_fact_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


# Summary

The Gold fact copies the governed Silver measure without changing NULL to zero. The regional population fact remains available for reconciliation and existing consumers.
